## Modelling the duration of delays of TunisAir

This notebook contains the steps for developing a Machine Learning model to predict delays of flights based on a number of predictors. The calculations use as input the data cleaned by the steps summarized in a [previous notebook of this project](/04_data-cleaning.ipynb).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Importing from .csv requires converting the dates to daytime in pandas
# Since duration calculations are already done, 
# we only need to parse the date-time fields for some plots later
df = pd.read_csv('data/Train_cleaned.csv', parse_dates=["STD_UTC", "STA_UTC"])
# Create specific time blocks
#def get_time_of_day(hour):
#    if 5 <= hour < 12:
#        return 'Morning'
#    elif 12 <= hour < 17:
#        return 'Afternoon'
#    elif 17 <= hour < 22:
#        return 'Evening'
#    else:
#        return 'Night'

#df['time_of_day'] = df['hour'].apply(get_time_of_day)
df["STD_UTC"].head()


### Train and test split

Given the sequential nature of this dataset, it is important to keep the order of records and not randomize the samples when creating splits in the data. In this way, the temporal structure of the time series is maintained. A gap of 200 flights was also included between partitions to reduce the effect of past and future data on a given partition.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import cross_validate

ts_cv = TimeSeriesSplit(
    n_splits=5,
    gap=200,
)


We also configured a particular implementation of the cross-validation algorithm to accommodate this training data split strategy when evaluating the models. Models were compared using the requested RMSE and two other informative metrics (MAE and R-squared).

In [ ]:
# define a custom function to evaluate the models with cross-validation
def evaluate(model, X, y, cv, model_prop=None, model_step=None):
    cv_results = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"],
        return_estimator=True,
    )
    if model_prop is not None:
        if model_step is not None:
            values = [
                getattr(m[model_step], model_prop) for m in cv_results["estimator"]
            ]
        else:
            values = [getattr(m, model_prop) for m in cv_results["estimator"]]
        print(f"Mean model.{model_prop} = {np.mean(values)}")
    mae = -cv_results["test_neg_mean_absolute_error"]
    rmse = -cv_results["test_neg_root_mean_squared_error"]
    rsquared = -cv_results["test_r2"]
    print(
        f"R-squared:     {rsquared.mean():.3f} +/- {rsquared.std():.3f}\n"
        f"Mean Absolute Error:     {mae.mean():.3f} +/- {mae.std():.3f}\n"
        f"Root Mean Squared Error: {rmse.mean():.3f} +/- {rmse.std():.3f}"
    )
    # To display the fitted estimator diagrams in the notebook.
    return cv_results["estimator"][0]

Similarly, a variation of the code was created to access the results when a TransformedTargetRegressor is used within a pipeline.

In [ ]:
import numpy as np
from sklearn.model_selection import cross_validate
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline

def evaluate_trans(model, X, y, cv, model_prop=None, model_step=None):
    cv_results = cross_validate(
        model, X, y, cv=cv, 
        scoring=["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"], 
        return_estimator=True
    )
    
    if model_prop is not None:
        values = []
        for m in cv_results["estimator"]:
            current = m
            
            # 1. If the outer model is the TransformedTargetRegressor wrapper
            if isinstance(current, TransformedTargetRegressor):
                current = current.regressor_
                
            # 2. If we are dealing with a Pipeline, extract the specific step
            if isinstance(current, Pipeline) and model_step is not None:
                # If the step name 'ridgecv' isn't found directly, look inside a wrapped step
                if model_step in current.named_steps:
                    current = current.named_steps[model_step]
                elif 'transformedtargetregressor' in current.named_steps:
                    step = current.named_steps['transformedtargetregressor']
                    if isinstance(step, TransformedTargetRegressor):
                        current = step.regressor_
            
            # 3. If the step itself was wrapped inside TransformedTargetRegressor
            if isinstance(current, TransformedTargetRegressor):
                current = current.regressor_
                
            values.append(getattr(current, model_prop))
            
        print(f"Mean model.{model_prop} = {np.mean(values):.5f}")

    rsquared = -cv_results["test_r2"]    
    mae = -cv_results["test_neg_mean_absolute_error"]
    rmse = -cv_results["test_neg_root_mean_squared_error"]
    print(f"R-squared: {rsquared.mean():.3f} +/- {rsquared.std():.3f}")
    print(f"Mean Absolute Error: {mae.mean():.3f} +/- {mae.std():.3f}")
    print(f"Root Mean Squared Error: {rmse.mean():.3f} +/- {rmse.std():.3f}")

### Definition of predictors and target subsets
After exploratory data analyses (see separate notebook) we identified a number of predictors to be included in our modelling approach. These predictors (features) are "duration", "interval", "rotation_sequence", "DEPSIZE", "year", "month","weekday", "hour", "weekend", "time_of_day", "DEPSTN", and "ARRSTN". To asses multi-collinearity of predictors, we also explore the correlations among numerical predictors and the target. Luckily, the numerical features are not strongly correlated among them but this might be obscured by any periodic cycles in the data.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
cor_data = df[["duration", "interval","rotation_sequence", "DEPSIZE", "target"]]

cor_data = cor_data.dropna()
plt.rcParams.update({'font.size': 10})
sns.heatmap(cor_data.corr(),vmin=-1.0, vmax=1.0, cmap="vlag", square=True,annot=True,fmt='.1f')


In [ ]:
# Defining target and predictors
df_small = df[["duration", "interval", "rotation_sequence", "DEPSIZE", "year", 
                     "month","weekday", "hour",
                    "weekend", "time_of_day", "DEPSTN", "ARRSTN", "target"]].dropna()

df_small = df_small[df_small['target'] <= 600] # removing extreme delays. Delays of more than 10 hours are extremely rare and hinder the model
X = df_small.drop("target", axis=1)
y = df_small["target"]

## Modelling

### Model 1: Linear regression

We start by modelling the delays using a ridge regression. This linear regression technique, commonly used to analyze data that suffers from multicollinearity, adds a L2 penalty (the squared magnitude of coefficients multiplied by a lambda/alpha hyperparameter) to the ordinary least squares cost function. This shrinks coefficients toward zero, reducing variance and preventing overfitting. Larger values of alpha specify stronger regularization and the variant we used (RidgeCV) estimates the optimal alpha parameter by cross-validation.

In [ ]:
# Accessory function to log-transform the target
from sklearn.linear_model import RidgeCV
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor

#1. Define target transformation functions
def func(x):
    return np.log1p(x)
def inverse_func(x):
    return np.expm1(x)
alphas = np.logspace(-6, 6, 25)
regr = TransformedTargetRegressor(regressor=RidgeCV(alphas = alphas),
                                  func=func,
                                  inverse_func=inverse_func,
                                  check_inverse=False)


This linear regression model considers a number of categorical and numeric features. All categorical features were dummy-coded and a robust transformation was applied to the numerical features. Among the categoric features are the ones representing the periodic events ("hour", "weekday", "month", "year") which are not accounted for with this dummy-coding strategy. 

In [ ]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline, make_pipeline

categorical_columns=["year", "month", "weekend", "time_of_day", "DEPSTN", "ARRSTN"]
numeric_columns=["duration", "interval", "rotation_sequence", "DEPSIZE"]
one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
transformer = RobustScaler()
alphas = np.logspace(-6, 6, 25)
ridge_regression =  make_pipeline(
    ColumnTransformer(
        transformers=[
            ("categorical", one_hot_encoder, categorical_columns),
            ("numeric", transformer, numeric_columns)
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    ),
   regr
)


ridge_regression.fit(X,y)

In [ ]:
evaluate_trans(
    ridge_regression, X, y, cv=ts_cv, model_prop="alpha_", model_step="ridgecv")


The variance explained by this model is poor (R<sup>2</sup> = 0.08) and the error is 81 minutes. We now will try other models.

### Model 2: Histogram-Based Boosted Regression Trees

Our next model uses the [Histogram-Based Boosted Regression Trees](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html) algorithm. Histogram-Based Boosted Regression Trees (HGBT or HistGB) are high-performance machine learning models that speed up traditional gradient boosting. They group continuous features into discrete bins (histograms), avoiding slow sorting steps during node splits.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
gbrt_categorical_columns=["year", "month", "weekend", "time_of_day", "DEPSTN", "ARRSTN"]
gbrt = HistGradientBoostingRegressor(categorical_features=gbrt_categorical_columns, 
                                     random_state=42)
evaluate(gbrt, X, y, cv=ts_cv, model_prop="n_iter_")

With its default settings, this model outperformed the ridge-regression model. The models RMSE is 70.3 minutes and explains 15% of the variance in the data (R<sup>2</sup> = 0.147). 

### Hyper parameter optimization of the HGBRT
The HGBRT model is the best performing model in terms of RMSE, so we attempt to optimize the hyperparameters of this model using a randomized grid search.

In [ ]:
import numpy as np
from scipy.stats import loguniform
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


param_grid = {
    'early_stopping': [True, False],
    'learning_rate': loguniform(0.01, 1),
    'interaction_cst': ['pairwise', 'no_interactions'],
    'l2_regularization': np.logspace(-6, 0, 6),
    'max_iter': [3, 10, 30, 100, 300, 1000],
    'max_leaf_nodes': [2, 5, 10, 20, 50, 100],
    'n_iter_no_change': np.arange(5,30),
    }


rs = RandomizedSearchCV(
    HistGradientBoostingRegressor(categorical_features=gbrt_categorical_columns, 
                                     random_state=42), 
    param_grid,
    scoring='neg_mean_squared_error',
    cv = ts_cv,
    verbose=0,
    n_jobs=-1,
    random_state=42,
)
#.drop(['time_of_day','DEPSTN','ARRSTN'], axis=1)
rs.fit(X,y)
# Get params
# Returning best score after GridSearch and the best parameter combination
print("                Best score:", rs.best_score_.round(4))
print("Best parameter combination:", rs.best_params_)
RSearch_best = rs.best_estimator_
# Best parameter combination: {'early_stopping': False, 'interaction_cst': 'pairwise',
#  'l2_regularization': np.float64(0.00025118864315095795), 'learning_rate': np.float64(0.0828891686688514), 
# 'max_iter': 300, 'max_leaf_nodes': 20, 'n_iter_no_change': np.int64(12)}


### Evaluate performance of the HGBRT model with optimized parameters
We now use the best performing parameters of the randomized grid search to implement and test the HGBRT model. Interestingly, this model alone performs better that when included in a pipeline with feature engineering. This is thus our winning model.

In [ ]:
#categorical_columns = categorical_columns.append(['DEPSTN'])
ogbrt = HistGradientBoostingRegressor(categorical_features=gbrt_categorical_columns,
                                    n_iter_no_change = 12,
                                    max_leaf_nodes = 20,
                                    max_iter = 300,
                                    learning_rate = 0.083,
                                    l2_regularization = 0.00025,
                                    interaction_cst = 'pairwise',
                                    early_stopping= False,
                                    random_state=42)
evaluate(ogbrt, X, y, cv=ts_cv, model_prop="n_iter_")

Interestingly, the optimization of the hyperparameters reduced the performance metrics by a small fraction but we assume this should allow the model to generalize better on future data.

### Model 3: Ridge regression with spline transformers to fit cyclic features
In a final attempt we implement a variation of the ridge regression model with a preprocessing step where we add different splines to model cyclic features like month and week days. This feature engineering step can help the model to predict cyclic peaks of delays.

In [ ]:
from sklearn.preprocessing import SplineTransformer


def periodic_spline_transformer(period, n_splines=None, degree=3):
    if n_splines is None:
        n_splines = period
    n_knots = n_splines + 1  # periodic and include_bias is True
    return SplineTransformer(
        degree=degree,
        n_knots=n_knots,
        knots=np.linspace(0, period, n_knots).reshape(n_knots, 1),
        extrapolation="periodic",
        include_bias=True,
    )


We use the engineered periodic splines along with categorical and continuous features in a pipeline that includes a log-transformation of the target.  

In [ ]:
regr = TransformedTargetRegressor(regressor=RidgeCV(alphas = alphas),
                                  func=func,
                                  inverse_func=inverse_func,
                                  check_inverse=False)

cyclic_spline_transformer = ColumnTransformer(
    transformers=[
        ("categorical", one_hot_encoder, ['DEPSTN', 'year', "time_of_day"]),
        ("cyclic_month", periodic_spline_transformer(12, n_splines=6), ["month"]),
        ("cyclic_weekday", periodic_spline_transformer(7, n_splines=3), ["weekday"]),
        #("cyclic_hour", periodic_spline_transformer(24, n_splines=12), ["hour"]),
        ("continuous", RobustScaler(), ["duration", "rotation_sequence"]) #"duration", 
    ],
    remainder="drop",
    verbose_feature_names_out=False
    )

cyclic_spline_linear_pipeline = make_pipeline(
    cyclic_spline_transformer,
    regr,
)

evaluate(cyclic_spline_linear_pipeline, X, y, cv=ts_cv)

### Adding interaction features to the previous model

We now add another feature engineering step to model the second-order interactions between "weekend" and "time-of-day factors". We apply the polynomial transformer in a separate step to restrict the interactions to these two features.

In [ ]:
from sklearn.preprocessing import StandardScaler
reduced_cyclic_spline_transformer = ColumnTransformer(
    transformers=[
        ("categorical", one_hot_encoder, ['year']),
        ("cyclic_month", periodic_spline_transformer(12, n_splines=6), ['month']),
        ("cyclic_weekday", periodic_spline_transformer(7, n_splines=3), ['weekday']),
        #("cyclic_hour", periodic_spline_transformer(24, n_splines=8), ["hour"]),
        ("continuous", StandardScaler(), ['duration'])
    ],
    remainder="drop",
    verbose_feature_names_out=False
    )

In [ ]:
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import PolynomialFeatures,FunctionTransformer, MinMaxScaler
tt_columns=["DEPSTN", "ARRSTN"]
target_encoder = TargetEncoder(target_type="continuous", shuffle=False)
interactions = make_pipeline(
    ColumnTransformer([
        ("weekend", FunctionTransformer(
                    lambda x: x == "True", feature_names_out="one-to-one"), ["weekend"]),
        ("time_of_day", one_hot_encoder, ['time_of_day']),
        ("target_encoded", target_encoder, tt_columns)
        ],
        verbose_feature_names_out=False,
        remainder="drop",
    ),
    PolynomialFeatures(degree=2, interaction_only=True, include_bias=True),
)

In [ ]:
regr = TransformedTargetRegressor(regressor=RidgeCV(alphas = alphas),
                                  func=func,
                                  inverse_func=inverse_func,
                                  check_inverse=False)
cyclic_spline_interactions_pipeline = make_pipeline(
    FeatureUnion(
        [
            ("marginal", reduced_cyclic_spline_transformer),
            ("interactions", interactions),
        ],
        verbose_feature_names_out=True,
    ).set_output(transform="pandas"),
    RidgeCV(alphas=alphas),
)
evaluate(cyclic_spline_interactions_pipeline, X, y, cv=ts_cv)

Even after the many feature engineering steps, this model under-performed the HGBRT model.

### Hurdle regression ensemble

We herein implemented a two-stage ensemble or hurdle regression. It is composed of two models: the first model predicts the occurrence of delays treating the target as binary (0: target >0, 1: target > 0), the second model uses these results and makes a quantitative prediction of the data subset where a target > 0 is predicted. 

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, PolynomialFeatures, RobustScaler

target_encoder = TargetEncoder(target_type="continuous")
preprocessor = ColumnTransformer([
    ("categorical", one_hot_encoder, 
                          ["weekend", "year", "month", "hour"]),
    ("numeric", transformer, ["duration", "interval", "DEPSIZE"]),
    
    ],
    remainder="drop",
    
    )
   
preprocessor.fit(X, y)

In [ ]:
interactions = make_pipeline(
    ColumnTransformer([
        ("weekend", FunctionTransformer(
                    lambda x: x == "True", feature_names_out="one-to-one"), ["weekend"]),
        ("time_of_day", one_hot_encoder, ['time_of_day']),
        ("target_encoded", target_encoder, ["DEPSTN", "ARRSTN", "AC"])
        ],
        verbose_feature_names_out=False,
        remainder="drop",
    ),
    PolynomialFeatures(degree=2, interaction_only=True, include_bias=True),
)

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import LogisticRegression, LinearRegression, RidgeCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler,MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted


# 2. Define the custom Hurdle Model (same as before)
class HurdleRegression(BaseEstimator, RegressorMixin):
    def __init__(self, classifier=None, regressor=None):
        self.classifier = classifier if classifier is not None else LogisticRegression()
        self.regressor = regressor if regressor is not None else LinearRegression()

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        y_binary = (y > 0).astype(int)
        self.classifier.fit(X, y_binary)
        
        nonzero_mask = y > 0
        if np.sum(nonzero_mask) > 0:
            self.regressor.fit(X[nonzero_mask], y[nonzero_mask])
        
        self.is_fitted_ = True
        return self

    def predict(self, X):
        check_is_fitted(self, 'is_fitted_')
        X = check_array(X)
        
        zero_preds = self.classifier.predict(X)
        final_preds = np.zeros(X.shape[0])  # Ensured length matches rows
        
        nonzero_mask = zero_preds == 1
        if np.sum(nonzero_mask) > 0:
            final_preds[nonzero_mask] = self.regressor.predict(X[nonzero_mask])
            final_preds[final_preds < 0] = 0  # Force non-negative predictions
            
        return final_preds


# 2. Build the scikit-learn Pipeline
hurdle_pipeline = Pipeline([
    ('preprocessing', preprocessor),                 # Step 1: Preprocess features
    ('hurdle_model', HurdleRegression(            # Step 2: Custom two-step model
        classifier=LogisticRegression(max_iter=5000),
        regressor=gbrt
    ))
])

# 4. Train and predict using the Pipeline
evaluate(hurdle_pipeline, X, y, cv=ts_cv)

This model also under-performed the HGBRT model. For the sake of simplicity, we excluded the results of this model in the final presentation. 

We have a clear winner in both RMSE and R<sup>2</sup>: the HGBRT model !!!

### Plotting the best model residuals

We first calculate the predicted values for all our candidate models.

In [ ]:
all_splits = list(ts_cv.split(X, y))
train_0, test_0 = all_splits[0]
last_hours = slice(-6800, -6600)
#sample_weights = np.where(y.iloc[train_0] > 80, 2.5, 1.0)

ridge_regression.fit(X.iloc[train_0], y.iloc[train_0])
linear_predictions = ridge_regression.predict(X.iloc[test_0])

ogbrt.fit(X.iloc[train_0], y.iloc[train_0])#, histgradientboostingregressor__sample_weight = sample_weights)
oHGBRT_predictions = np.clip(ogbrt.predict(X.iloc[test_0]), a_min=0, a_max=None)

cyclic_spline_linear_pipeline.fit(X.iloc[train_0], y.iloc[train_0])
cyclic_spline_predictions = np.clip(cyclic_spline_linear_pipeline.predict(X.iloc[test_0]),
                                    a_min=0, a_max=None)


#hurdle_pipeline.fit(X.iloc[train_0], y.iloc[train_0])
#hurdle_predictions = np.clip(hurdle_pipeline.predict(X.iloc[test_0]),
#                             a_min=0, a_max=None)


In [ ]:
# Plot 2: Residuals vs Predicted Delays
import seaborn as sns
import matplotlib.pyplot as plt
cyresiduals = y.iloc[test_0] - oHGBRT_predictions
ax2 = sns.scatterplot(x=y.iloc[test_0], y=cyresiduals, alpha=0.4, color="crimson")
ax2.axhline(y=0, color="black", linestyle="--", linewidth=2)
ax2.set_title("Residual Error Plot HGBRT", fontsize=14)
ax2.set_xlabel("Predicted Delay (Minutes)", fontsize=12)
ax2.set_ylabel("Residuals (Actual - Predicted)", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
df.iloc[train_0][last_hours].STA_UTC.min() # We use these dates for plotting

### Plotting the predictions of the three main models

In [ ]:
ogbrt.fit(X.iloc[train_0], y.iloc[train_0])
oHGBRT_predictions = ogbrt.predict(X.iloc[test_0])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.iloc[train_0][last_hours].STA_UTC, y.iloc[test_0].values[last_hours],
    "x-", alpha=0.2, label="Actual delays", color="black")
ax.plot(df.iloc[train_0][last_hours].STA_UTC, oHGBRT_predictions[last_hours],
    "x-", label="HGBRT model", color="blue")
ax.plot(df.iloc[train_0][last_hours].STA_UTC, cyclic_spline_predictions[last_hours],
    "x-", label="Ridge Regression with splines", color="orange")
ax.plot(df.iloc[train_0][last_hours].STA_UTC, linear_predictions[last_hours], 
    "x-", label="Ridge Regression", color="green")

_ = ax.legend()
_ = ax.set(
    title="delays",
    xticks=pd.date_range(start="2017-07-08", end="2017-08-21", freq="7D"),
    xlabel="Date",
    ylabel="delay (minutes)",
)
fig.autofmt_xdate()
plt.show()

In [ ]:
# Summary of model metrics
rmse = pd.DataFrame()
rmse["model"] = ["ridge_regression","ridge_regression_splines","HGBRT"]
rmse["RMSE"] = [80.06,76.93,71.50]
rmse["r2"] = [0.04, 0.00, 0.12]
rmse["sd"] = [13.01,7.827, 3.82]
rmse

In the above table we can clearly see that the HGBRT outperforms the other two models and the difference is appreciable in both RMSE and R<sup>2</sup>.

In [ ]:

ax = rmse.plot.bar(x='model', 
                y="RMSE",
                yerr="sd")
ax.set_title("Root mean squared errors of models")
ax.set_ylabel("RMSE (min.)")


### Estimating feature importance in predicting delays
We used a permutation test to estimate the importance of each feature. This technique involves randomly shuffling the values of a single feature and observing the resulting degradation of the model’s score (in this case RMSE). By breaking the relationship between the feature and the target, we determine how much the model relies on such particular feature.

In [ ]:
from sklearn.inspection import permutation_importance
test_arr = np.concatenate([x[1] for x in all_splits]) # recovering all 5 test partition indexes
raw_importances = permutation_importance(
    ogbrt, X.iloc[test_arr], y.iloc[test_arr], scoring= "neg_root_mean_squared_error", n_repeats=10, random_state=42, n_jobs=-1
) 
feature_names = X.columns #[f"feature {i}" for i in range(X.shape[1])]
HGBRT_importances = pd.Series(raw_importances.importances_mean, index=feature_names).sort_values(ascending=False)

#oHGBRT_importances = pd.Series(ogbrt.feature_importances_, index=X.iloc[train_0].columns)
#oHGBRT_importances_sorted_idx = np.argsort(ogbrt.feature_importances_)

fig, ax = plt.subplots()
HGBRT_importances.plot.bar(yerr=raw_importances.importances_std, ax=ax)
ax.set_title("Feature importances in the HGBRT model")
ax.set_ylabel("Mean RMSE decrease")
fig.tight_layout()
plt.show()


### Airports with most delays

After the interval between successive flights, the departure airport was the second most important feature in predicting the delays. We identify the top five delaying airports as those with more flights delayed by 30 or more minutes and subsequently plotted their mean delay durations in a barplot. 

In [ ]:
DEP_delays = df.query('target > 30')[['DEPSTN','target']].groupby(by='DEPSTN').count()
top_DEP_delays = DEP_delays.sort_values(by='target',ascending=False).head(5)

df_plot = df[df['DEPSTN'].isin(list(top_DEP_delays.index))]

In [ ]:

p = sns.catplot(
    data=df_plot.query('target > 0'), kind="bar",
    x="DEPSTN", y="target",# hue="sex",
    #errorbar="sd", #palette="dark", alpha=.6, height=6
)
p.despine(left=True)
p.set_axis_labels("", "average duration of delays (min.)")
p.fig.suptitle("Top five airports in nr. of delays")

## Save final model to predict delay duration on test

In [ ]:
import joblib

joblib.dump(
    ogbrt,
    "models/final_hgbrt_model.pkl"
)